# Dia 17 — Gerando documentos

**🃏 Cartas:** 17 · 38 · 59 &nbsp;·&nbsp; **📘 Original:** *Day 17. Generating documents*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 17 — O relatório preliminar

O telefone da Iara toca às 16h50. Ela escuta, diz "sim, doutora" três vezes e desliga.

— A Delegada Rita quer um **relatório preliminar** amanhã cedo. Nada de dossiê final ainda: só o que
já sabemos, bem organizado, para ela pedir os dados aos bancos.

Você abre um editor de texto e começa a copiar números das saídas dos últimos dias. A Iara põe a mão
sobre o seu teclado.

— Pare. Se você copiar à mão, vai errar um centavo. E quando chegar a denúncia número 38, vai ter que
refazer tudo. Relatório de investigação é **produto de dados**: o script lê as fontes, calcula e
escreve. Rodou de novo, saiu igual. Isso é reprodutibilidade, e é ela que te defende quando alguém
perguntar "de onde saiu esse número?".

Ela aponta para os hashes que você gerou na sala de evidências.

— E coloque isso no rodapé. Quem receber o relatório tem que conseguir provar que as evidências não
mudaram. Formato? A doutora imprime tudo. Mas a planilha vai para o cartório, e lá o Excel é em
português. Pense nisso.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-17-documentos")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Gerar DOCX, PDF e XLSX com as bibliotecas do curso original.
- Entender o que mudou do `fpdf` antigo para o `fpdf2`.
- Gerar relatórios em **Markdown** e **HTML** só com a biblioteca padrão.
- Exportar CSV que abre certo no **Excel em português** (`;`, `utf-8-sig`, vírgula decimal).
- Incluir hash SHA-256 e carimbo de tempo no relatório.
- Entender a estrutura interna de um PDF escrevendo um à mão.

## 🧠 Conceito em 2 minutos

Um relatório automático tem sempre três partes: **ler** as fontes, **calcular** os números e
**escrever** o documento. Só a terceira muda de formato para formato:

| Formato | Biblioteca | Quando usar |
|---|---|---|
| `.md` (Markdown) | nenhuma (texto puro) | GitHub, Obsidian, notas internas |
| `.html` | nenhuma (texto puro) | relatório bonito, imprimível como PDF pelo navegador |
| `.csv` | `csv` (nativa) | dados para planilha |
| `.docx` | `python-docx` | quando o destinatário vai **editar** no Word |
| `.xlsx` | `XlsxWriter` / `openpyxl` | planilha com fórmulas e formatação |
| `.pdf` | `fpdf2` (ou navegador) | documento final, que não deve ser alterado |

**O Excel brasileiro** espera `;` como separador (porque a vírgula é a casa decimal) e só reconhece
UTF-8 se o arquivo começar com o **BOM** (três bytes invisíveis, `EF BB BF`). Em Python, isso é só
`encoding="utf-8-sig"`.

**Reprodutibilidade:** o mesmo script com os mesmos dados gera os mesmos números. Junte a isso o
**SHA-256** de cada evidência e o **carimbo de data/hora** com fuso, e qualquer pessoa consegue
auditar o seu relatório.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/create_xlsx.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-17-documentos/original/create_xlsx.py) | Planilha de funcionários com fórmula de média (`XlsxWriter`) |
| [`original/create_docx.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-17-documentos/original/create_docx.py) | Documento Word com títulos, parágrafo, quebra de página e imagem (`python-docx`) |
| [`original/create_pdf.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-17-documentos/original/create_pdf.py) | PDF com uma linha de texto e uma imagem (`fpdf`) |
| [`original/2026_create_pdf.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-17-documentos/original/2026_create_pdf.py) | **Atualização 2026:** o mesmo PDF sem os parâmetros obsoletos do fpdf2 |
| [`original/histogram.png`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-17-documentos/original/histogram.png) | Imagem usada pelos scripts de DOCX e PDF |

```bash
pip install XlsxWriter python-docx fpdf2
cd dia-17-documentos/original      # os scripts procuram histogram.png na pasta atual
python create_xlsx.py
python create_docx.py
python create_pdf.py
python 2026_create_pdf.py
```

> ⚠️ **fpdf × fpdf2.** O PDF do curso manda instalar `fpdf`, um pacote parado desde 2017. Hoje use
> **`fpdf2`**, que continua com `from fpdf import FPDF`. O `create_pdf.py` original roda no fpdf2,
> mas avisa que `txt=` virou `text=` e que `ln=` está obsoleto (use `new_x`/`new_y`). Não instale
> os dois ao mesmo tempo: eles brigam pelo mesmo nome `fpdf`.

> 🔍 **Ache o bug do original:** em `create_xlsx.py`, a fórmula sai `=average(B1:B4)`, deixando o
> último funcionário fora da média (e incluindo o cabeçalho B1, que o Excel ignora por ser texto).
> Qual seria o intervalo certo?

O autor lembra que também dá para gerar apresentações com `python-pptx`.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q XlsxWriter python-docx fpdf2

### `original/2026_create_pdf.py` — Gerar um PDF com fpdf2 (versão atualizada em 2026)

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Gerar um PDF com fpdf2 (versão atualizada em 2026)
# ATUALIZAÇÃO 2026 feita para este repositório, BASEADA no original de cipher387
# (Day_17/create_pdf.py, MIT © 2023 Cyb_detective). O original continua intacto em
# create_pdf.py. Aqui só trocamos o que o fpdf2 marcou como obsoleto:
#   • txt=  → text=          (renomeado na versão 2.7.6)
#   • ln=2  → new_x / new_y  (ln está obsoleto desde a versão 2.5.2)
# Requer: pip install fpdf2   (rode dentro de original/, onde está o histogram.png)
# ──────────────────────────────────────────────────────────────────────────

# 💬 O fpdf2 mantém o mesmo nome de importação do pacote antigo.
from fpdf import FPDF
# 💬 XPos e YPos dizem para onde o "cursor" vai depois de escrever uma célula.
from fpdf.enums import XPos, YPos

pdfFile = FPDF()

pdfFile.add_page()

# 💬 "Helvetica" é o nome oficial da fonte embutida ("Arial" é só um apelido).
pdfFile.set_font("Helvetica", size=12)

# 💬 ln=2 do original significava "vá para a linha de baixo, mantendo a margem esquerda".
# 💬 No fpdf2 isso se escreve: new_x=XPos.LEFT, new_y=YPos.NEXT.
pdfFile.cell(20, 10, text="Report text", new_x=XPos.LEFT, new_y=YPos.NEXT, align="C")

pdfFile.image("histogram.png", 10, 40, 30)

pdfFile.output("report.pdf")

### `original/create_docx.py` — Gerar um documento Word (.docx) com python-docx

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Gerar um documento Word (.docx) com python-docx
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_17/create_docx.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install python-docx  (rode dentro de original/, onde está o histogram.png)
# ──────────────────────────────────────────────────────────────────────────
# 💬 O pacote se instala como 'python-docx', mas importa como 'docx'.
from docx import Document
# 💬 Inches converte polegadas para a unidade interna do Word.
from docx.shared import Inches

# 💬 Cria um documento em branco, na memória (ainda não existe arquivo).
document = Document()

# 💬 Nível 0 = título principal do documento.
document.add_heading('Report', 0)

# 💬 Nível 1 = título de seção (o 'Título 1' do Word).
document.add_heading('Report', level=1)


# 💬 Um parágrafo de texto comum.
document.add_paragraph(
    'Some text in report'
)

# 💬 Quebra de página: o que vier depois começa na página 2.
document.add_page_break()

# 💬 Insere a imagem com 1,25 polegada de largura (≈3,2 cm).
document.add_picture('histogram.png', width=Inches(1.25))

# 💬 Só agora o arquivo é gravado no disco.
document.save('report.docx')

### `original/create_pdf.py` — Gerar um PDF com FPDF

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Gerar um PDF com FPDF
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_17/create_pdf.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install fpdf2  (o pacote antigo "fpdf" parou em 2017; o fpdf2 importa igual)
# ──────────────────────────────────────────────────────────────────────────
# 💬 Tanto o fpdf antigo quanto o fpdf2 usam 'from fpdf import FPDF'.
from fpdf import FPDF


# 💬 Cria o PDF (A4, retrato, medidas em milímetros por padrão).
pdfFile = FPDF()

# 💬 Todo PDF precisa de pelo menos uma página antes de escrever.
pdfFile.add_page()
 
# 💬 'Arial' é apelido da fonte embutida Helvetica. As fontes embutidas
# 💬 NÃO têm acentos fora do Latin-1 — para português completo, use add_font com um .ttf.
pdfFile.set_font("Arial", size = 12)

# 💬 Uma 'célula' de 20 x 10 mm com o texto centralizado. No fpdf2, txt= e ln= são OBSOLETOS:
# 💬 ainda funcionam, mas mostram aviso. Veja a versão atual em 2026_create_pdf.py.
pdfFile.cell(20, 10, txt = "Report text", ln = 2, align = 'C')

# 💬 Imagem em x=10 mm, y=40 mm, com 30 mm de largura.
pdfFile.image('histogram.png', 10, 40, 30)

# 💬 Grava o arquivo no disco.
pdfFile.output("report.pdf") 

### `original/create_xlsx.py` — Gerar uma planilha Excel (.xlsx) com XlsxWriter

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Gerar uma planilha Excel (.xlsx) com XlsxWriter
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_17/create_xlsx.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install XlsxWriter
# ──────────────────────────────────────────────────────────────────────────
# 💬 XlsxWriter só ESCREVE arquivos .xlsx (para ler, use openpyxl).
import xlsxwriter


# 💬 O 'livro' (arquivo) e, abaixo, uma aba (planilha) vazia.
workbook = xlsxwriter.Workbook('employees.xlsx')
worksheet = workbook.add_worksheet()

# 💬 Uma tupla de listas: cada lista é uma linha [nome, idade]. A primeira é o cabeçalho.
employees = (
    ['Name', 'Age'],
    ['John Smith', 33],
    ['Eric Gold',   26],
    ['Simon Silver',  37],
    ['James Conor',    50],
)

# 💬 Contador de linha. No XlsxWriter a contagem começa em 0 (linha 0 = linha 1 do Excel).
row = 0

# 💬 Desempacota cada linha em duas variáveis e escreve na coluna 0 (A) e 1 (B).
for name, age in (employees):
    worksheet.write(row, 0, name)
    worksheet.write(row, 1, age)
    row += 1


# 💬 Depois do laço, 'row' aponta para a primeira linha livre.
worksheet.write(row, 0, 'Average age')
# 💬 Uma fórmula do Excel montada como texto. Atenção: row-1 faz o intervalo terminar
# 💬 uma linha antes do último funcionário (B1:B4) — confira no Excel e tente corrigir!
# 💬 Ah, e no Excel em português a função aparece como MÉDIA, mas o arquivo guarda 'AVERAGE'.
worksheet.write(row, 1, '=average(B1:B'+str(row-1)+')')

# 💬 Fechar é obrigatório: é aqui que o .xlsx é realmente gravado.
workbook.close()

## 🇧🇷 Nível 2 — Camada Brasil

Só **biblioteca padrão**, 100% offline (dados do caso).

| Arquivo | O que faz |
|---|---|
| [`br/01_relatorio_markdown.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-17-documentos/br/01_relatorio_markdown.py) | Relatório preliminar em `.md`: resumo, tabela por chave PIX, produtos, próximos passos |
| [`br/02_relatorio_html.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-17-documentos/br/02_relatorio_html.py) | Relatório HTML com cartões de totais, tabela das denúncias, SHA-256 das evidências e carimbo America/Sao_Paulo |
| [`br/03_csv_para_excel_br.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-17-documentos/br/03_csv_para_excel_br.py) | CSV certo e CSV errado lado a lado; proteção contra CSV injection; resumo por mês |
| [`br/04_pdf_na_unha.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-17-documentos/br/04_pdf_na_unha.py) | **Bônus:** um PDF de 1 página escrito byte a byte, com autoverificação da tabela xref |

```bash
cd dia-17-documentos
python br/02_relatorio_html.py     # depois abra saidas/relatorio_preliminar.html
python br/04_pdf_na_unha.py
```

**O que você deve ver no caso:** 37 denúncias, **R$ 54.877,40** de prejuízo, 10 em outubro e 27 em
novembro. A chave da Jéssica Prado Lima é a que mais recebeu (16 denúncias, R$ 21.242,10).

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_relatorio_markdown.py` — Relatório preliminar do caso em Markdown (.md), gerado por código.

In [ ]:
%cd -q {PASTA_NB}
"""
Relatório preliminar do caso em Markdown (.md), gerado por código.

Carta 38 (rank A) — Jutsu do Relatório Automático

O que você aprende:
  • a juntar números de várias fontes num texto pronto para ler;
  • a escrever tabelas em Markdown (o formato do GitHub, do Obsidian e do Notion);
  • a formatar dinheiro em reais (R$ 54.877,40) e datas em dd/mm/aaaa;
  • por que um relatório gerado por script é reprodutível (e um copia-e-cola não é).

Como rodar (dentro da pasta dia-17-documentos):
    python br/01_relatorio_markdown.py
"""

import csv
import json
from collections import Counter, defaultdict
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal          # dinheiro sem erro de arredondamento
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
ANALISTA = "Analista NFA"  #@param {type:"string"}
NUMERO_DO_RELATORIO = "NFA-2025-117"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"

# Fuso de Brasília. No Windows pode faltar a base de fusos (tzdata): usamos -03:00 fixo.
try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO = timezone(timedelta(hours=-3))


def reais(valor):
    """Decimal('54877.4') → 'R$ 54.877,40' (ponto no milhar, vírgula no centavo)."""
    texto = f"{valor:,.2f}"                    # '54,877.40' (padrão americano)
    # Trocamos em três tempos para não confundir os símbolos:
    return "R$ " + texto.replace(",", "§").replace(".", ",").replace("§", ".")


# ── 1. Ler as fontes ──────────────────────────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    denuncias = list(csv.DictReader(arquivo, delimiter=";"))
ficha = json.loads((PASTA_CASO / "ficha_do_caso.json").read_text(encoding="utf-8"))
rdap = json.loads((PASTA_CASO / "rdap_megaoferta.json").read_text(encoding="utf-8"))

# ── 2. Calcular os números do relatório ───────────────────────────────────
total = sum(Decimal(d["valor"]) for d in denuncias)
datas = sorted(date.fromisoformat(d["data"]) for d in denuncias)
eventos = {e["eventAction"]: e["eventDate"] for e in rdap["events"]}
criado = datetime.fromisoformat(eventos["registration"].replace("Z", "+00:00")).date()

por_chave = defaultdict(lambda: [0, Decimal("0")])   # chave → [quantidade, soma]
for d in denuncias:
    por_chave[d["chave_pix_destino"]][0] += 1
    por_chave[d["chave_pix_destino"]][1] += Decimal(d["valor"])
titular = {c["chave"]: c["titular"] for c in ficha["contas_laranja"]}

produtos = Counter(d["produto"] for d in denuncias)

# ── 3. Escrever o Markdown ────────────────────────────────────────────────
# Uma lista de linhas que depois juntamos com "\n". Mais fácil de ler que um textão.
agora = datetime.now(FUSO)
md = [
    f"# Relatório preliminar {NUMERO_DO_RELATORIO} — Operação Vitrine Fantasma",
    "",
    f"**Gerado em:** {agora:%d/%m/%Y %H:%M} (horário de Brasília)  ",
    f"**Responsável:** {ANALISTA}  ",
    "**Classificação:** uso interno · dados fictícios de estudo",
    "",
    "## 1. Resumo",
    "",
    f"- **{len(denuncias)} denúncias** recebidas entre {datas[0]:%d/%m/%Y} e {datas[-1]:%d/%m/%Y}.",
    f"- **Prejuízo declarado:** {reais(total)}.",
    f"- **Domínio:** `{ficha['dominio']}`, criado em {criado:%d/%m/%Y} "
    f"(**{(datas[0] - criado).days} dias** antes da primeira denúncia).",
    f"- **CNPJ do rodapé:** {ficha['cnpj_rodape']} (dígito verificador inválido, ver Dia 2).",
    "",
    "## 2. Para onde foi o dinheiro (chaves PIX)",
    "",
    "| Chave PIX | Titular | Denúncias | Valor |",       # cabeçalho da tabela
    "|---|---|---:|---:|",                               # ---: alinha à direita
]
for chave, (qtd, soma) in sorted(por_chave.items(), key=lambda item: -item[1][1]):
    md.append(f"| `{chave[:8]}…` | {titular.get(chave, '?')} | {qtd} | {reais(soma)} |")

md += ["", "## 3. Produtos anunciados", "", "| Produto | Denúncias |", "|---|---:|"]
for produto, qtd in produtos.most_common():
    # O "|" é o separador de colunas; se aparecer no texto, escapamos com "\|".
    md.append(f"| {produto.replace('|', chr(92) + '|')} | {qtd} |")

md += [
    "",
    "## 4. Próximos passos",
    "",
    "- [ ] Ofício às instituições de pagamento das três chaves PIX.",
    "- [ ] Pedido de dados cadastrais do domínio ao registrador.",
    "- [ ] Mapa das vítimas por cidade (Dia 18) e linha do tempo (Dia 19).",
    "",
    "---",
    "*Relatório gerado automaticamente por `br/01_relatorio_markdown.py`. "
    "Rodar de novo com os mesmos dados produz os mesmos números.*",
]

SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / "relatorio_preliminar.md"
destino.write_text("\n".join(md) + "\n", encoding="utf-8")

print("\n".join(md[:16]))
print("…")
print(f"\n💾 {destino.relative_to(PASTA_DIA)} gravado ({len(md)} linhas). Abra no VS Code, GitHub ou Obsidian.")

### `br/02_relatorio_html.py` — Relatório em HTML com tabela, totais, hashes SHA-256 e carimbo de tempo.

In [ ]:
%cd -q {PASTA_NB}
"""
Relatório em HTML com tabela, totais, hashes SHA-256 e carimbo de tempo.

Carta 38 (rank A) — Jutsu do Relatório Automático (parte 2)

O que você aprende:
  • a montar uma página HTML só com strings (sem instalar nada);
  • a escapar o texto com html.escape (um nome com "<" não pode quebrar a página);
  • a incluir o SHA-256 de cada evidência no próprio relatório (cadeia de custódia, Dia 14);
  • a gerar PDF sem biblioteca: abrir no navegador e "Imprimir → Salvar como PDF".

Como rodar (dentro da pasta dia-17-documentos):
    python br/02_relatorio_html.py
    (depois abra saidas/relatorio_preliminar.html no navegador)
"""

import csv
import hashlib
import html                          # html.escape: transforma < > & " em texto seguro
from datetime import datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
TITULO = "Operação Vitrine Fantasma — relatório preliminar"  #@param {type:"string"}
ANALISTA = "Analista NFA"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"

try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:                    # Windows sem tzdata
    FUSO = timezone(timedelta(hours=-3))


def reais(valor):
    return "R$ " + f"{valor:,.2f}".replace(",", "§").replace(".", ",").replace("§", ".")


def sha256(caminho):
    """Hash SHA-256 lendo em blocos de 64 KB (funciona até com arquivos enormes)."""
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(65536), b""):
            h.update(bloco)
    return h.hexdigest()


e = html.escape   # apelido curto: vamos usar muito

# ── 1. Dados ──────────────────────────────────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    denuncias = sorted(csv.DictReader(arquivo, delimiter=";"), key=lambda d: d["data"])
total = sum(Decimal(d["valor"]) for d in denuncias)

evidencias = sorted(p for p in PASTA_CASO.rglob("*") if p.is_file())

# ── 2. Linhas da tabela de denúncias ──────────────────────────────────────
linhas_denuncias = []
for d in denuncias:
    dia, mes, ano = d["data"][8:10], d["data"][5:7], d["data"][:4]
    linhas_denuncias.append(
        "<tr>"
        f"<td>{e(d['id'])}</td><td>{dia}/{mes}/{ano}</td>"
        f"<td>{e(d['cidade'])}</td><td>{e(d['produto'])}</td>"
        f"<td class='num'>{reais(Decimal(d['valor']))}</td>"
        f"<td><code>{e(d['chave_pix_destino'][:8])}…</code></td>"
        "</tr>"
    )
    # Repare: o NOME e o CPF das vítimas ficam FORA do relatório. Minimização (LGPD):
    # quem lê o relatório preliminar não precisa deles para entender o golpe.

linhas_hash = [
    f"<tr><td>{e(p.relative_to(PASTA_CASO).as_posix())}</td>"
    f"<td class='num'>{p.stat().st_size:,}".replace(",", ".") + " B</td>"
    f"<td><code>{sha256(p)}</code></td></tr>"
    for p in evidencias
]

agora = datetime.now(FUSO)

# ── 3. A página ───────────────────────────────────────────────────────────
# Chaves {{ }} duplas: dentro de uma f-string, "{{" vira "{" literal (necessário no CSS).
pagina = f"""<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<title>{e(TITULO)}</title>
<style>
  body {{ font-family: system-ui, sans-serif; max-width: 960px; margin: 2rem auto; padding: 0 1rem; color: #1d2433; }}
  h1 {{ border-bottom: 3px solid #1f5fa8; padding-bottom: .4rem; }}
  .cards {{ display: flex; gap: 1rem; flex-wrap: wrap; }}
  .card {{ flex: 1; min-width: 180px; background: #eef3fa; border-radius: 8px; padding: .8rem 1rem; }}
  .card b {{ display: block; font-size: 1.6rem; color: #1f5fa8; }}
  table {{ border-collapse: collapse; width: 100%; margin: 1rem 0; font-size: .9rem; }}
  th, td {{ border-bottom: 1px solid #d5dbe5; padding: .35rem .5rem; text-align: left; }}
  th {{ background: #1f5fa8; color: white; }}
  .num {{ text-align: right; white-space: nowrap; }}
  code {{ font-size: .8rem; word-break: break-all; }}
  footer {{ margin-top: 2rem; font-size: .8rem; color: #5b6475; }}
  @media print {{ th {{ color: black; background: #ddd; }} .card {{ border: 1px solid #999; }} }}
</style>
</head>
<body>
<h1>{e(TITULO)}</h1>
<p>Gerado em <b>{agora:%d/%m/%Y %H:%M:%S}</b> (horário de Brasília, UTC{agora:%z}) · Responsável: {e(ANALISTA)}</p>

<div class="cards">
  <div class="card"><b>{len(denuncias)}</b>denúncias</div>
  <div class="card"><b>{reais(total)}</b>prejuízo declarado</div>
  <div class="card"><b>{len({d['chave_pix_destino'] for d in denuncias})}</b>chaves PIX de destino</div>
  <div class="card"><b>{len(evidencias)}</b>arquivos de evidência</div>
</div>

<h2>Denúncias (ordem cronológica)</h2>
<table>
<tr><th>ID</th><th>Data</th><th>Cidade (como digitada)</th><th>Produto</th><th>Valor</th><th>Chave PIX</th></tr>
{chr(10).join(linhas_denuncias)}
</table>

<h2>Integridade das evidências (SHA-256)</h2>
<p>Qualquer pessoa pode recalcular estes hashes e confirmar que os arquivos não mudaram desde a geração deste relatório.</p>
<table>
<tr><th>Arquivo</th><th>Tamanho</th><th>SHA-256</th></tr>
{chr(10).join(linhas_hash)}
</table>

<footer>Relatório gerado por <code>br/02_relatorio_html.py</code> · Dados fictícios de estudo · Para PDF: Ctrl+P → Salvar como PDF.</footer>
</body>
</html>
"""

SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / "relatorio_preliminar.html"
destino.write_text(pagina, encoding="utf-8")
print(f"🧾 {len(denuncias)} denúncias · {reais(total)} · {len(evidencias)} evidências com hash")
print(f"💾 {destino.relative_to(PASTA_DIA)} gravado ({len(pagina):,} caracteres).".replace(",", "."))
print("🖨️ Abra no navegador e use Imprimir → Salvar como PDF.")

# No Colab/Jupyter (onde não existe __file__), mostra a página logo abaixo da célula.
if "__file__" not in globals():
    try:
        from IPython.display import HTML, display
        display(HTML(pagina))
    except ImportError:
        pass

### `br/03_csv_para_excel_br.py` — CSV que abre certo no Excel brasileiro: ";", utf-8-sig e vírgula decimal.

In [ ]:
%cd -q {PASTA_NB}
"""
CSV que abre certo no Excel brasileiro: ";", utf-8-sig e vírgula decimal.

Carta 38 (rank A) — Jutsu do Relatório Automático (parte 3)

O que você aprende:
  • por que um CSV "normal" (vírgula, UTF-8 sem BOM) abre bagunçado no Excel em português;
  • o que é o BOM do utf-8-sig e por que ele salva os acentos;
  • a escrever números com vírgula decimal (1.899,00) e datas dd/mm/aaaa;
  • a evitar a "injeção de fórmula" (CSV injection) ao exportar dados de terceiros.

Como rodar (dentro da pasta dia-17-documentos):
    python br/03_csv_para_excel_br.py
"""

import csv
from collections import defaultdict
from decimal import Decimal
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
GERAR_VERSAO_ERRADA = True  #@param {type:"boolean"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"


def numero_br(valor):
    """Decimal('1899.00') → '1899,00' (sem separador de milhar: o Excel entende melhor)."""
    return f"{valor:.2f}".replace(".", ",")


def data_br(iso):
    """'2025-11-28' → '28/11/2025'."""
    ano, mes, dia = iso.split("-")
    return f"{dia}/{mes}/{ano}"


def seguro(texto):
    """Protege contra CSV injection: célula que começa com = + - @ vira texto.

    Se uma 'vítima' digitar =HIPERLINK("http://site-malicioso"...) no relato, o Excel
    executaria como fórmula. Um apóstrofo na frente faz o Excel tratar como texto.
    """
    return "'" + texto if texto[:1] in ("=", "+", "-", "@") else texto


# ── 1. Ler as denúncias ───────────────────────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    denuncias = list(csv.DictReader(arquivo, delimiter=";"))

SAIDAS.mkdir(exist_ok=True)

# ── 2. (Didático) a versão ERRADA, para você comparar no Excel ───────────
if GERAR_VERSAO_ERRADA:
    with open(SAIDAS / "denuncias_ERRADO.csv", "w", newline="", encoding="utf-8") as f:
        escritor = csv.writer(f)                       # vírgula, sem BOM, ponto decimal
        escritor.writerow(["id", "data", "cidade", "produto", "valor"])
        for d in denuncias:
            escritor.writerow([d["id"], d["data"], d["cidade"], d["produto"], d["valor"]])
    print("❌ saidas/denuncias_ERRADO.csv: no Excel pt-BR tudo cai numa coluna só e 'Divinópolis' vira 'DivinÃ³polis'.")

# ── 3. A versão CERTA ─────────────────────────────────────────────────────
# utf-8-sig grava 3 bytes invisíveis no começo (EF BB BF, o "BOM"). É o sinal que o
# Excel procura para saber que o arquivo está em UTF-8. Sem ele, o Excel chuta Windows-1252.
with open(SAIDAS / "denuncias_excel_br.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.writer(f, delimiter=";")         # ";" porque no Brasil a vírgula é decimal
    escritor.writerow(["ID", "Data", "Cidade", "Produto", "Valor (R$)", "Relato"])
    for d in denuncias:
        escritor.writerow([d["id"], data_br(d["data"]), seguro(d["cidade"]),
                           seguro(d["produto"]), numero_br(Decimal(d["valor"])), seguro(d["relato_curto"])])
print("✅ saidas/denuncias_excel_br.csv: colunas separadas, acentos certos, valores somáveis.")

# ── 4. Uma tabela-resumo por mês (pronta para virar gráfico no Excel) ────
por_mes = defaultdict(lambda: [0, Decimal("0")])
for d in denuncias:
    mes = d["data"][:7]                              # '2025-11'
    por_mes[mes][0] += 1
    por_mes[mes][1] += Decimal(d["valor"])

NOMES_MES = ["", "janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho",
             "agosto", "setembro", "outubro", "novembro", "dezembro"]
with open(SAIDAS / "resumo_por_mes.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.writer(f, delimiter=";")
    escritor.writerow(["Mês", "Denúncias", "Prejuízo (R$)"])
    for mes in sorted(por_mes):
        ano, m = mes.split("-")
        qtd, soma = por_mes[mes]
        escritor.writerow([f"{NOMES_MES[int(m)]}/{ano}", qtd, numero_br(soma)])
        print(f"   {NOMES_MES[int(m)]:>9}/{ano}: {qtd:>2} denúncias · R$ {numero_br(soma)}")
print("✅ saidas/resumo_por_mes.csv gravado.")

# ── 5. Prova de que o BOM está lá ─────────────────────────────────────────
primeiros = (SAIDAS / "denuncias_excel_br.csv").read_bytes()[:3]
print(f"\n🔬 Primeiros 3 bytes do arquivo certo: {primeiros.hex(' ').upper()}  (EF BB BF = BOM do UTF-8)")
print("💡 Ao ler esse arquivo de volta no Python, use encoding='utf-8-sig' para o BOM não grudar no 1º cabeçalho.")

### `br/04_pdf_na_unha.py` — Bônus: um PDF de verdade escrito "na unha", só com a biblioteca padrão.

In [ ]:
%cd -q {PASTA_NB}
"""
Bônus: um PDF de verdade escrito "na unha", só com a biblioteca padrão.

Carta 38 (rank A) — Jutsu do Relatório Automático (bônus)

O que você aprende:
  • que um PDF é, por dentro, um arquivo de texto com "objetos" numerados;
  • o papel da tabela xref (o índice com a posição em bytes de cada objeto);
  • a escrever texto com acentos usando a fonte embutida Helvetica + WinAnsiEncoding;
  • a conferir se o arquivo gerado é um PDF válido (cabeçalho %PDF, xref e %%EOF).

Para relatórios de verdade, prefira fpdf2 ou o "Imprimir como PDF" do navegador (script 02).
Este exercício é para você nunca mais achar que PDF é mágica.

Como rodar (dentro da pasta dia-17-documentos):
    python br/04_pdf_na_unha.py
"""

import csv
from decimal import Decimal
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
TITULO = "Operação Vitrine Fantasma"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"


def texto_pdf(texto):
    """Prepara um texto para dentro de ( ) no PDF.

    1. Parênteses e barra invertida têm significado especial: escapamos com "\\".
    2. A fonte embutida usa WinAnsiEncoding, que é praticamente o cp1252 do Windows:
       por isso codificamos em cp1252 (é ele que tem "ã", "ç", "é"...).
    Caracteres que não existem no cp1252 (emojis, por exemplo) viram "?".
    """
    texto = texto.replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")
    return texto.encode("cp1252", errors="replace")


def data_br(iso):
    """'2025-10-12' → '12/10/2025'."""
    return "/".join(reversed(iso.split("-")))


# ── 1. Conteúdo: algumas linhas de resumo do caso ─────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    denuncias = list(csv.DictReader(arquivo, delimiter=";"))
total = sum(Decimal(d["valor"]) for d in denuncias)
total_br = f"{total:,.2f}".replace(",", "§").replace(".", ",").replace("§", ".")

linhas = [
    (18, TITULO),
    (12, "Relatório preliminar — Núcleo de Fontes Abertas (dados fictícios)"),
    (12, ""),
    (12, f"Denúncias recebidas: {len(denuncias)}"),
    (12, f"Prejuízo declarado: R$ {total_br}"),
    (12, f"Período: {data_br(min(d['data'] for d in denuncias))} a {data_br(max(d['data'] for d in denuncias))}"),
    (12, "Pagamento exclusivamente por PIX, três chaves de destino."),
    (12, ""),
    (10, "Este PDF foi escrito byte a byte por br/04_pdf_na_unha.py (só stdlib)."),
]

# ── 2. O "fluxo de conteúdo": os comandos de desenho da página ────────────
# BT/ET = começa/termina texto · Tf = fonte e tamanho · Td = move o cursor · Tj = escreve
# A origem (0,0) do PDF é o canto INFERIOR esquerdo. A4 = 595 x 842 pontos.
comandos = [b"BT", b"/F1 18 Tf", b"56 780 Td"]
tamanho_anterior = 18
for i, (tamanho, texto) in enumerate(linhas):
    if i > 0:
        comandos.append(f"0 -{int(tamanho_anterior * 1.6)} Td".encode())  # desce uma linha
    comandos.append(f"/F1 {tamanho} Tf".encode())
    comandos.append(b"(" + texto_pdf(texto) + b") Tj")
    tamanho_anterior = tamanho
comandos.append(b"ET")
# Uma linha horizontal azul embaixo do título (re = retângulo, f = preencher)
comandos.append(b"0.12 0.37 0.66 rg 56 770 483 2 re f")
conteudo = b"\n".join(comandos)

# ── 3. Os objetos do PDF ──────────────────────────────────────────────────
objetos = [
    b"<< /Type /Catalog /Pages 2 0 R >>",                                  # 1: a raiz
    b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",                           # 2: lista de páginas
    b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 595 842] "                # 3: a página A4
    b"/Resources << /Font << /F1 4 0 R >> >> /Contents 5 0 R >>",
    b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica "                # 4: a fonte
    b"/Encoding /WinAnsiEncoding >>",
    b"<< /Length " + str(len(conteudo)).encode() + b" >>\nstream\n"         # 5: o desenho
    + conteudo + b"\nendstream",
]

# ── 4. Montar o arquivo, anotando a posição (em bytes) de cada objeto ─────
# O comentário com 4 bytes acima de 127 avisa aos programas que o arquivo é binário.
pdf = bytearray(b"%PDF-1.4\n%\xe2\xe3\xcf\xd3\n")
posicoes = []
for numero, corpo in enumerate(objetos, start=1):
    posicoes.append(len(pdf))                      # onde este objeto começa
    pdf += f"{numero} 0 obj\n".encode() + corpo + b"\nendobj\n"

# A tabela xref: uma linha de EXATAMENTE 20 bytes por objeto (10 dígitos, espaço, 5 dígitos...)
inicio_xref = len(pdf)
pdf += f"xref\n0 {len(objetos) + 1}\n".encode()
pdf += b"0000000000 65535 f \n"                    # o objeto 0 é sempre esse "livre"
for pos in posicoes:
    pdf += f"{pos:010d} 00000 n \n".encode()
pdf += f"trailer\n<< /Size {len(objetos) + 1} /Root 1 0 R >>\nstartxref\n{inicio_xref}\n%%EOF\n".encode()

SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / "relatorio_na_unha.pdf"
destino.write_bytes(bytes(pdf))

# ── 5. Autoverificação: o arquivo se parece com um PDF válido? ────────────
dados = destino.read_bytes()
ok_cabecalho = dados.startswith(b"%PDF-")
ok_fim = dados.rstrip().endswith(b"%%EOF")
xref_declarado = int(dados.rsplit(b"startxref", 1)[1].split()[0])
ok_xref = dados[xref_declarado:xref_declarado + 4] == b"xref"
ok_objetos = all(dados[p:].startswith(f"{n} 0 obj".encode()) for n, p in enumerate(posicoes, 1))

print(f"📄 {destino.relative_to(PASTA_DIA)} · {len(dados)} bytes")
print(f"   {'✅' if ok_cabecalho else '❌'} começa com %PDF-")
print(f"   {'✅' if ok_xref else '❌'} startxref aponta para a palavra 'xref'")
print(f"   {'✅' if ok_objetos else '❌'} cada posição da xref aponta para o objeto certo")
print(f"   {'✅' if ok_fim else '❌'} termina com %%EOF")
if not (ok_cabecalho and ok_fim and ok_xref and ok_objetos):
    raise SystemExit("❌ O PDF gerado está inconsistente.")
print("🎉 Abra o arquivo no navegador ou no leitor de PDF: os acentos aparecem certinhos.")

## ⚖️ Ética, LGPD e OPSEC

- **Minimização.** O relatório HTML mostra cidade, produto e valor, mas **não** nome nem CPF das
  vítimas: para entender o golpe, quem lê não precisa deles. Dado pessoal só entra no documento que
  realmente precisa dele (o ofício ao banco, por exemplo).
- **Metadados vazam.** DOCX e PDF guardam autor, empresa e datas nas propriedades do arquivo. Antes
  de enviar para fora, confira (Arquivo → Propriedades) e limpe o que não deve sair.
- **CSV injection.** Uma célula que começa com `=`, `+`, `-` ou `@` vira fórmula no Excel. Se o texto
  veio de terceiros (relatos, comentários raspados), proteja com um apóstrofo, como no script 03.
- **Relatório é documento oficial.** Diferencie claramente **fato** ("a chave X recebeu 16 PIX") de
  **hipótese** ("o titular pode ser laranja"). E nunca edite o número à mão depois de gerado.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Por que gravar o CSV com encoding='utf-8-sig' e delimiter=';' para o Excel brasileiro?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Porque o Excel não abre UTF-8", "B) O BOM avisa o Excel que é UTF-8 e o ';' evita conflito com a vírgula decimal", "C) Para o arquivo ficar menor", "D) Porque o Python exige"]
correta = "B) O BOM avisa o Excel que é UTF-8 e o ';' evita conflito com a vírgula decimal"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Sem BOM o Excel chuta Windows-1252 (acentos quebram); com vírgula como separador, os decimais viram colunas.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. No fpdf2, o que aconteceu com os parâmetros txt= e ln= do curso original?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Foram removidos e o script quebra", "B) txt virou text e ln foi substituído por new_x/new_y; os antigos ainda funcionam com aviso", "C) Continuam iguais", "D) Agora são obrigatórios"]
correta = "B) txt virou text e ln foi substituído por new_x/new_y; os antigos ainda funcionam com aviso"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "São obsoletos (DeprecationWarning), não removidos. A versão 2026_create_pdf.py usa a forma nova.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Para que serve a tabela xref dentro de um PDF?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Guardar a senha do arquivo", "B) Listar as fontes usadas", "C) Dizer em que posição (em bytes) começa cada objeto do arquivo", "D) Guardar o histórico de edições"]
correta = "C) Dizer em que posição (em bytes) começa cada objeto do arquivo"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O leitor de PDF vai ao startxref, lê a tabela e pula direto para cada objeto.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 17: Fábrica de ofícios

**Carta 59 · Rank S**

## A missão

Com o relatório preliminar em mãos, a Delegada Rita vai oficiar as instituições de pagamento que
receberam os PIX. São três chaves, três instituições, e a Iara não quer **ninguém** digitando ofício.

Escreva um script que:

1. Agrupe as denúncias de `denuncias.csv` por `chave_pix_destino`.
2. Descubra titular e instituição de cada chave em `ficha_do_caso.json`.
3. A partir de **um único modelo** (use `string.Template`, da biblioteca padrão), gere **um ofício
   HTML por chave**, com número sequencial, data por extenso em português ("12 de outubro de 2026"),
   tabela das transações daquela chave e total em reais.
4. Grave os ofícios em `saidas/oficios/` e um `indice_oficios.csv` (`;`, `utf-8-sig`) com número,
   instituição, quantidade, total e **SHA-256** de cada ofício.
5. Confira: a soma dos três ofícios tem que dar o total das denúncias.

## Critérios de rank S

- [ ] Só biblioteca padrão, 100% offline
- [ ] Um modelo só; nenhum texto de ofício repetido no código
- [ ] Todo texto vindo dos dados passa por `html.escape`
- [ ] Data por extenso sem depender do `locale` do sistema
- [ ] A soma confere com R$ 54.877,40

## Dica

`Template("Olá, $nome").substitute(nome="Rita")` troca `$nome`. Prefira `substitute` a
`safe_substitute`: se faltar uma variável, ele dá erro, e um erro é melhor do que um ofício com
"$titular" escrito no meio.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 17 — Fábrica de ofícios.

Para cada chave PIX que recebeu dinheiro, gera um ofício em HTML (um arquivo por instituição)
a partir de UM modelo com string.Template, com a lista de transações daquela chave.
No fim, grava um índice com o SHA-256 de cada ofício gerado.
100% offline: usa só caso/dados/.
"""

import csv
import hashlib
import html
import json
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path
from string import Template          # modelo com $variaveis, da biblioteca padrão

NUMERO_BASE = 300  #@param {type:"integer"}
CIDADE_EMISSAO = "Belo Horizonte"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas" / "oficios"

try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO = timezone(timedelta(hours=-3))

MESES = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho",
         "agosto", "setembro", "outubro", "novembro", "dezembro"]

MODELO = Template("""<!doctype html>
<html lang="pt-BR"><head><meta charset="utf-8"><title>Ofício $numero</title>
<style>body{font-family:Georgia,serif;max-width:720px;margin:2rem auto;line-height:1.5}
table{border-collapse:collapse;width:100%}td,th{border:1px solid #999;padding:.3rem .5rem}
.num{text-align:right}</style></head><body>
<p><b>NÚCLEO DE FONTES ABERTAS (fictício)</b><br>Ofício nº $numero/2025 — Operação Vitrine Fantasma</p>
<p>$cidade, $data_extenso.</p>
<p>À <b>$banco</b><br>Setor de atendimento a autoridades</p>
<p><b>Assunto:</b> solicitação de dados cadastrais e de movimentação da chave PIX <code>$chave</code>.</p>
<p>Em apoio à investigação conduzida pela Delegada Rita Assunção, solicitamos os dados cadastrais
do(a) titular <b>$titular</b> e o extrato das operações abaixo, que somam <b>$total</b> em
<b>$quantidade</b> transferências declaradas por vítimas.</p>
<table><tr><th>Data</th><th>Denúncia</th><th>Valor</th></tr>
$linhas
</table>
<p>Atenciosamente,<br><br>__________________________<br>Analista do NFA</p>
</body></html>
""")


def reais(valor):
    return "R$ " + f"{valor:,.2f}".replace(",", "§").replace(".", ",").replace("§", ".")


# ── Dados ────────────────────────────────────────────────────────────────
ficha = json.loads((PASTA_CASO / "ficha_do_caso.json").read_text(encoding="utf-8"))
contas = {c["chave"]: c for c in ficha["contas_laranja"]}

por_chave = defaultdict(list)
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    for d in csv.DictReader(arquivo, delimiter=";"):
        por_chave[d["chave_pix_destino"]].append(d)

hoje = datetime.now(FUSO)
data_extenso = f"{hoje.day} de {MESES[hoje.month - 1]} de {hoje.year}"
SAIDAS.mkdir(parents=True, exist_ok=True)

indice = []
for n, (chave, transacoes) in enumerate(sorted(por_chave.items()), start=NUMERO_BASE):
    conta = contas.get(chave, {"titular": "(desconhecido)", "banco": "(instituição a identificar)"})
    transacoes.sort(key=lambda d: d["data"])
    linhas = "\n".join(
        f"<tr><td>{'/'.join(reversed(d['data'].split('-')))}</td><td>{html.escape(d['id'])}</td>"
        f"<td class='num'>{reais(Decimal(d['valor']))}</td></tr>"
        for d in transacoes
    )
    total = sum(Decimal(d["valor"]) for d in transacoes)
    # substitute() exige TODAS as variáveis: se faltar uma, dá erro (melhor que um ofício incompleto).
    documento = MODELO.substitute(
        numero=n, cidade=html.escape(CIDADE_EMISSAO), data_extenso=data_extenso,
        banco=html.escape(conta["banco"]), titular=html.escape(conta["titular"]),
        chave=html.escape(chave), total=reais(total), quantidade=len(transacoes), linhas=linhas,
    )
    nome = f"oficio_{n}_{chave[:8]}.html"
    (SAIDAS / nome).write_text(documento, encoding="utf-8")
    hash_doc = hashlib.sha256(documento.encode("utf-8")).hexdigest()
    indice.append({"oficio": n, "arquivo": nome, "banco": conta["banco"], "titular": conta["titular"],
                   "transacoes": len(transacoes), "total": f"{total:.2f}".replace(".", ","), "sha256": hash_doc})
    print(f"📨 Ofício {n}: {conta['banco']:<30} {len(transacoes):>2} transações · {reais(total):>13}")

with open(SAIDAS / "indice_oficios.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=list(indice[0]), delimiter=";")
    escritor.writeheader()
    escritor.writerows(indice)

soma = sum(Decimal(i["total"].replace(",", ".")) for i in indice)
print(f"\n✅ {len(indice)} ofícios · soma {reais(soma)} (confere com o total das denúncias)")
print("💾 saidas/oficios/ + indice_oficios.csv (com SHA-256 de cada ofício)")

## 🔎 Inovação OSINT do dia

**HTML → PDF com WeasyPrint.** Se você já tem o relatório em HTML (como o do script 02), a biblioteca
**WeasyPrint** (`pip install weasyprint`, licença BSD) converte para PDF respeitando o CSS, inclusive
regras de impressão como quebra de página e cabeçalho/rodapé (`@page`). Uma linha:
`from weasyprint import HTML; HTML("relatorio.html").write_pdf("relatorio.pdf")`. Assim você mantém
**um só modelo** (HTML) e gera os dois formatos. No Windows ela depende das bibliotecas do Pango;
siga o guia de instalação oficial.

## 📚 Para ir além

- fpdf2 (documentação e guia de migração do fpdf antigo): <https://py-pdf.github.io/fpdf2/>
- python-docx: <https://python-docx.readthedocs.io/>
- XlsxWriter: <https://xlsxwriter.readthedocs.io/>
- Especificação do PDF 1.7 (ISO 32000-1), cópia pública da Adobe: <https://opensource.adobe.com/dc-acrobat-sdk-docs/pdfstandards/PDF32000_2008.pdf>
- OWASP sobre CSV injection: <https://owasp.org/www-community/attacks/CSV_Injection>
- WeasyPrint: <https://doc.courtbouillon.org/weasyprint/>

## ✅ Checklist

- [ ] Gerei XLSX, DOCX e PDF com os scripts originais
- [ ] Rodei o `2026_create_pdf.py` sem avisos de obsoleto
- [ ] Achei o bug da fórmula de média do `create_xlsx.py`
- [ ] Abri o CSV certo e o errado no Excel (ou LibreOffice) e vi a diferença
- [ ] Gerei o relatório HTML e salvei como PDF pelo navegador
- [ ] Abri o PDF feito "na unha"
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 17](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)